**Materia:** Tecnologías Emergentes — Primavera 2026  
**Profesor:** Mtro. Rafael Pérez Aguirre

# 🧠 Memory (Memoria conversacional)

Por defecto, un LLM no recuerda nada de mensajes anteriores — cada invocación es independiente. Pero para construir chatbots o asistentes útiles, necesitamos que el modelo tenga **contexto de la conversación**: que recuerde lo que dijiste antes y pueda referenciarlo en respuestas futuras.

En LangChain, la "memoria" no es más que **mantener y pasar el historial de mensajes** al modelo en cada turno. Veremos cómo hacerlo manualmente, con chains, y con agentes persistentes usando LangGraph.

📄 [Documentación oficial: Memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)

## Configuración

Cargar y/o verificar las variables de entorno necesarias.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

variables_requeridas = ["GEMINI_API_KEY"]
for var in variables_requeridas:
    if os.getenv(var):
        print(f"✅ {var} cargada correctamente")
    else:
        print(f"❌ {var} no encontrada")

✅ GEMINI_API_KEY cargada correctamente


## Historia de conversación manual

La forma más simple de agregar memoria es mantener una **lista de mensajes** que crece con cada turno. En cada invocación le pasamos toda la lista al modelo — así tiene contexto completo de lo que se ha dicho.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

modelo = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

# Historial de mensajes — empieza con el system prompt
historial = [
    SystemMessage(content="Eres un asistente amable. Responde siempre en español y de forma concisa.")
]

In [3]:
# Primer turno
pregunta_1 = "Hola, me llamo Puma y estudio ingeniería en sistemas computacionales en la IBERO Puebla."
historial.append(HumanMessage(content=pregunta_1))

respuesta_1 = modelo.invoke(historial)
historial.append(respuesta_1)  # guardar la respuesta del modelo también

print(f"Usuario: {pregunta_1}")
print(f"Asistente: {respuesta_1.content}")

Usuario: Hola, me llamo Puma y estudio ingeniería en sistemas computacionales en la IBERO Puebla.
Asistente: [{'type': 'text', 'text': '¡Hola, Puma! Mucho gusto. Es una excelente carrera y la IBERO Puebla es una gran institución. ¿En qué puedo ayudarte hoy con tus estudios o cualquier otro tema?', 'extras': {'signature': 'EtkECtYEAb4+9vtLmOnydv2/+t0D1iEkt1d22XF/EELKxl654Sa60IpfdO8iXH5S7V7IkXy+Ie92kfCF01nqRhokm/X/wEUL6tAIQ5k9iyTXe7qGCnn6CFtk/hBwVCQ7VfBgHOJWvMm9iU08gZYtVZPE7a2125lX4GsIhl25qYY4Tvw20226qWudYrPFKuhP415krPUkv0Xwf8S1JTrEbqXEpdPG53vgrJJAX9zOcr8GKOm/yHxbMizO11934QHlLM3oPlZCA1+zw1AVXdPVTY2ufyo5nWe4+io8MVUDSzIVadUkSZv5lCXM2CY6LRWurtfDiFBVh2fYOm07curprGqc4qvFX2/fSlhhGbSAxrH/5IqLO80xy6h60ssq5rJ6i9gIpGPiEcErFVY4t/2yohETmOdBhl+ZUnxCPhvOuG6ZjHwFDDri2t6bJzEgHrDwm0dauy16ZF2Td2uroQnnmpxVBMnKOiaUvQQvuXAW1hlthy1dTL14+HR6vYRKHCgJ981/Co+v6s6oDtZw7tCSaO1xV7Zf3gSULB9cyCy99B4pDxBXTpmLstp/8L9C7X3aYq2SZiREFdgDlcLkkveoOP2bukxxHZJ2z7b4tkXp39GDVfkVxkFn6TJk8+P8VH/je1HGrCEub8qECFLzKHnvFpYHFp0kg66EId9u

In [4]:
# Segundo turno — el modelo debe recordar el nombre
pregunta_2 = "¿Recuerdas cómo me llamo y qué estudio?"
historial.append(HumanMessage(content=pregunta_2))

respuesta_2 = modelo.invoke(historial)
historial.append(respuesta_2)

print(f"Usuario: {pregunta_2}")
print(f"Asistente: {respuesta_2.content}")

Usuario: ¿Recuerdas cómo me llamo y qué estudio?
Asistente: [{'type': 'text', 'text': '¡Claro que sí! Te llamas **Puma** y estudias **Ingeniería en Sistemas Computacionales** en la **IBERO Puebla**. ¿En qué puedo apoyarte hoy?', 'extras': {'signature': 'EoMDCoADAb4+9vsTdrIKYWi0jNDxXDdXZ+OIa5aWvEqjPiqpDBqKtR8bo8OWoRg7+qrI6uMGAsBM5oScJiU1UzZuhQl16pGzaJTXmAe4d5Hxb5oZOoZcKW0v7d/F2GToBWAtUp7xs6TXkPI95eHmbJnvk/kwsnW3JPhy6yAUN+xIc/N6GbCCkrl4XOkF3csG7qLzuYD9EJ/T9AvixwxWGmHE48CAfbTyTL5L3NNgkWaSx58bWsrjDtL4iIPBnlpaBFO/tP05PB10e7oxgv8BL+yOpRKkojtCLKwJ+Xbd4ruS90XdFOuNtKKYrXwGR5LRJw78gubfhsIZRRvP27NmQAoR1ozFuz+Ri9UcW0Wiz2ezd9h9wRqUyLQz2TLB9BaaQu6uI6LdrdZKJr5sqCETrJZvasaCj66I9YevaiOKykstFD1E1l0HrUJf+w7SFMwgeGcOy1XoF+YDyqd0/Ncb+5ouszsxdk9fRTstNG0q70BkAaTT4pmskXOm9B06fJsk3jk0eYAC'}}]


In [5]:
# Inspeccionar el historial completo
print(f"Total de mensajes en el historial: {len(historial)}\n")
for msg in historial:
    tipo = type(msg).__name__
    print(f"[{tipo}] {msg.content[:80]}..." if len(msg.content) > 80 else f"[{tipo}] {msg.content}")

Total de mensajes en el historial: 5

[SystemMessage] Eres un asistente amable. Responde siempre en español y de forma concisa.
[HumanMessage] Hola, me llamo Puma y estudio ingeniería en sistemas computacionales en la IBERO...
[AIMessage] [{'type': 'text', 'text': '¡Hola, Puma! Mucho gusto. Es una excelente carrera y la IBERO Puebla es una gran institución. ¿En qué puedo ayudarte hoy con tus estudios o cualquier otro tema?', 'extras': {'signature': 'EtkECtYEAb4+9vtLmOnydv2/+t0D1iEkt1d22XF/EELKxl654Sa60IpfdO8iXH5S7V7IkXy+Ie92kfCF01nqRhokm/X/wEUL6tAIQ5k9iyTXe7qGCnn6CFtk/hBwVCQ7VfBgHOJWvMm9iU08gZYtVZPE7a2125lX4GsIhl25qYY4Tvw20226qWudYrPFKuhP415krPUkv0Xwf8S1JTrEbqXEpdPG53vgrJJAX9zOcr8GKOm/yHxbMizO11934QHlLM3oPlZCA1+zw1AVXdPVTY2ufyo5nWe4+io8MVUDSzIVadUkSZv5lCXM2CY6LRWurtfDiFBVh2fYOm07curprGqc4qvFX2/fSlhhGbSAxrH/5IqLO80xy6h60ssq5rJ6i9gIpGPiEcErFVY4t/2yohETmOdBhl+ZUnxCPhvOuG6ZjHwFDDri2t6bJzEgHrDwm0dauy16ZF2Td2uroQnnmpxVBMnKOiaUvQQvuXAW1hlthy1dTL14+HR6vYRKHCgJ981/Co+v6s6oDtZw7tCSaO1xV7Zf3gSULB

Cada turno agrega **dos mensajes** al historial: el `HumanMessage` con la pregunta y el `AIMessage` con la respuesta. El modelo recibe el historial completo en cada invocación, por eso puede recordar información de turnos anteriores.

## Memoria en cadenas con `MessagesPlaceholder`

Cuando usamos chains (`prompt | llm | parser`), podemos inyectar el historial en el prompt usando `MessagesPlaceholder`. Esto nos permite construir una cadena conversacional reutilizable con una variable `historial` que se rellena en cada invocación.

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt_conversacional = ChatPromptTemplate.from_messages([
    ("system", "Eres un tutor de Python. Responde en español de forma clara y con ejemplos de código cuando sea útil."),
    MessagesPlaceholder(variable_name="historial"),  # aquí se inyecta el historial
    ("human", "{pregunta}")
])

cadena = prompt_conversacional | modelo | StrOutputParser()

In [7]:
# Primer turno — historial vacío
historial_chain = []

pregunta = "¿Qué es una lista por comprensión en Python?"
respuesta = cadena.invoke({"historial": historial_chain, "pregunta": pregunta})

# Guardar en el historial
historial_chain.append(HumanMessage(content=pregunta))
historial_chain.append(AIMessage(content=respuesta))

print(f"Usuario: {pregunta}")
print(f"Tutor: {respuesta}")

Usuario: ¿Qué es una lista por comprensión en Python?
Tutor: Una **lista por comprensión** (*list comprehension*) es una forma concisa y elegante de crear listas en Python. Permite generar una nueva lista aplicando una expresión a cada elemento de un iterable (como una lista, rango o tupla) en una sola línea de código.

Es una de las características favoritas de los programadores de Python porque suele ser más rápida y fácil de leer que un bucle `for` tradicional.

---

### 1. La estructura básica

La sintaxis general es:

```python
nueva_lista = [expresion for elemento in iterable]
```

#### Ejemplo comparativo:
Imagina que quieres crear una lista con los cuadrados de los números del 0 al 4.

**Usando un bucle `for` tradicional:**
```python
cuadrados = []
for x in range(5):
    cuadrados.append(x**2)

print(cuadrados) # Resultado: [0, 1, 4, 9, 16]
```

**Usando lista por comprensión:**
```python
cuadrados = [x**2 for x in range(5)]

print(cuadrados) # Resultado: [0, 1, 4, 9, 16]
```



In [8]:
# Segundo turno — la cadena recuerda la respuesta anterior
pregunta_2 = "¿Puedes mostrarme un ejemplo con números pares del 1 al 20?"
respuesta_2 = cadena.invoke({"historial": historial_chain, "pregunta": pregunta_2})

historial_chain.append(HumanMessage(content=pregunta_2))
historial_chain.append(AIMessage(content=respuesta_2))

print(f"Usuario: {pregunta_2}")
print(f"Tutor: {respuesta_2}")

Usuario: ¿Puedes mostrarme un ejemplo con números pares del 1 al 20?
Tutor: ¡Claro que sí! Vamos a crear una lista con los números pares del 1 al 20.

Para este caso, usaremos la función `range(1, 21)`. Recuerda que en Python, el límite superior del rango no se incluye, por eso usamos **21** para que llegue hasta el **20**.

### El código:

```python
# Lista por comprensión para obtener pares del 1 al 20
pares = [n for n in range(1, 21) if n % 2 == 0]

print(pares)
```

### ¿Qué está pasando aquí?
Si desglosamos la línea de código, verás que es muy lógica:

1.  **`n`**: Es lo que queremos guardar en la lista (el número tal cual).
2.  **`for n in range(1, 21)`**: Es el ciclo que recorre los números del 1 al 20.
3.  **`if n % 2 == 0`**: Es el filtro. Solo deja pasar a `n` si el resto de dividirlo por 2 es cero (es decir, si es par).

---

### Comparación con el método tradicional
Para que veas cuánto espacio ahorras, así se haría con un bucle normal:

```python
pares = []
for n in range(

In [9]:
# Tercer turno — referencia implícita a lo anterior
pregunta_3 = "¿Y si quisiera filtrar solo los divisibles entre 3?"
respuesta_3 = cadena.invoke({"historial": historial_chain, "pregunta": pregunta_3})

print(f"Usuario: {pregunta_3}")
print(f"Tutor: {respuesta_3}")

Usuario: ¿Y si quisiera filtrar solo los divisibles entre 3?
Tutor: Es exactamente la misma lógica. Solo debemos cambiar la condición dentro del `if`. 

Para saber si un número es divisible entre 3, usamos el operador módulo (`%`), que nos da el resto de la división. Si `n % 3 == 0`, significa que el número es divisible por 3.

### El código:

```python
# Lista de números del 1 al 20 divisibles entre 3
divisibles_3 = [n for n in range(1, 21) if n % 3 == 0]

print(divisibles_3)
# Resultado: [3, 6, 9, 12, 15, 18]
```

---

### ¿Y si quieres combinar condiciones?
Las listas por comprensión son muy potentes porque puedes usar operadores lógicos como `and` u `or`.

Por ejemplo, si quisieras los números que son **divisibles entre 2 Y también entre 3** (es decir, múltiplos de 6):

```python
multiplos_6 = [n for n in range(1, 21) if n % 2 == 0 and n % 3 == 0]

print(multiplos_6)
# Resultado: [6, 12, 18]
```

### Reto para ti:
¿Cómo crees que sería la lista por comprensión si quisiéramos los nú

Con `MessagesPlaceholder`, la cadena no cambia — solo cambia el historial que le pasamos. Esto hace que sea fácil reutilizar la misma cadena para múltiples conversaciones independientes.

## Memoria persistente con agentes (LangGraph)

Con `create_agent` podemos activar memoria persistente usando un **checkpointer** de LangGraph. Esto permite que el agente recuerde el historial entre invocaciones sin que tengamos que gestionarlo manualmente. Cada conversación se identifica con un `thread_id`.

In [10]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver guarda el historial en memoria RAM (ideal para demos y labs)
memoria = MemorySaver()

agente = create_agent(
    ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[],
    system_prompt="Eres un asistente personal. Recuerda los datos del usuario y úsalos en tus respuestas. Responde en español.",
    checkpointer=memoria
)

# El thread_id identifica una conversación específica
config = {"configurable": {"thread_id": "sesion-001"}}

In [11]:
# Primer mensaje — el agente guarda automáticamente el historial
resultado_1 = agente.invoke(
    {"messages": [{"role": "user", "content": "Hola, me llamo Antia y soy doctora."}]},
    config=config
)
print(resultado_1["messages"][-1].content)

[{'type': 'text', 'text': '¡Hola, Antia! Es un placer saludarte. He tomado nota de que eres doctora; es un gusto contar con esa información para asistirte mejor de ahora en adelante.\n\nComo tu asistente personal, estoy aquí para ayudarte en lo que necesites, ya sea organizando tu agenda profesional, investigando temas médicos, redactando documentos o cualquier otra tarea de tu día a día.\n\n¿En qué puedo ayudarte hoy?', 'extras': {'signature': 'EssKCsgKAb4+9vu+WtTgFrZ+VGuxUOUQo4H8+OzjxG/lDg2GlHTO6fD1AXF7kTSSwiv6hf1772gr/smO21xM3epu7+DClk2Konz/gbymnhd0XNILlsYGw0t7weHfnWgEc24gbDdMSsAT+Gj6WUVqmQups+fRzMlL5D5yKOlfHoaOF46rLtXmD3d3SXHEzoSbZ1L7SoD+/pdCfbXx49QDUTNx6Z5qhrRj9G9rjKEmGTKyKkwb8gFCpnu+INEhJudgVfIAKCQIPd+nU4hOWXewrP5RZiOGC3iZ1osehnjWGGzP7VofH0iN/ENKRESYcuSorVRVLB5WL6h9YqfbkvZx/HrMLOA/JeYhCn2fuXFQ2RA/8EBSmfcMi54dYD2Cj5yj1CH6cv/PYg6HiiiOS/J9rSg034n6ndp0QsWQ8dLO7FMtT+M9ncUCmWmAcjb0tEbWTA+s7wdbmMA6puJIW9/fVR1ph6IVswDJnwG+3GCosEhVCwOAl51AGhMdmCjznoU2YgUHd1ch3OW8qPsfaGe6E+PSNEEU0Uw1IbCg+k

In [12]:
# Segundo mensaje — mismo thread_id, el agente recuerda sin que nosotros pasemos el historial
resultado_2 = agente.invoke(
    {"messages": [{"role": "user", "content": "¿Sabes a qué me dedico?"}]},
    config=config
)
print(resultado_2["messages"][-1].content)

[{'type': 'text', 'text': '¡Claro que sí, Antia! Recuerdo perfectamente que me dijiste que eres **doctora**. \n\n¿Hay algo específico relacionado con tu profesión o con tu día a día en lo que te pueda ayudar hoy?', 'extras': {'signature': 'EpIDCo8DAb4+9vuqQUnDkEMs79YHB10lGj+4zx29a0RCLP0PG817EdxZtelDRQ9ORTYVQ7VlnslNWKbH3/VKE0OIHw+sPtjQSyS3P7nPBnWjlOXSvIhLqTAKK5lYTXD0WEC/LNvv+cuP97LqbFT8hTdpWxZMds3MSVCAZMdn7ggAVZzh5L0gIA/MoZLhI0rqvXOLG80iI7kPrY3tSK+iTcYpKpjB5rpwMYDjOlW4EuXlCmB64qimxzqPMaHOPZ8P1xmAyDzLAXqlEJaTyGX4FDSVjOvVMFnIGbjx5XbwwZa0BAaVLVXEetHKEjUnoETX2Ks9qgd2ybYv4Sx2D8HdjRmv9bpDQQM9PxkL1EZ3Cr5O53jUXsTqmsv+BQFgRK/1KCx3poEzIYZwEjxRuPr3wgBYgj7iF+IgQWKicFaQ2095g2C3pRovE/gHnoX0OjDbjSiMyHsHRxk2suq5LE/dHqh8rgYxQnyFDqqNAd+UFJ2uiabTAxTGV1MPKfgNIruhBvY160ozslTDCcDIBQbUov3BSQXd'}}]


In [13]:
# Conversación distinta — thread_id diferente, sin memoria del hilo anterior
config_nueva = {"configurable": {"thread_id": "sesion-002"}}

resultado_3 = agente.invoke(
    {"messages": [{"role": "user", "content": "¿Sabes cómo me llamo?"}]},
    config=config_nueva
)
print(resultado_3["messages"][-1].content)

[{'type': 'text', 'text': 'Aún no me has dicho tu nombre. Como soy tu asistente personal, me encantaría saberlo para recordarlo y dirigirme a ti de forma más cercana en nuestras futuras conversaciones.\n\n¿Cómo te llamas?', 'extras': {'signature': 'Es4JCssJAb4+9vs0q7s+piEq9w3Aqh/ZBLJdLg+MHEQCqXmfFBXkOQs/dfjuQyrkrWJO44xh0A+fzEPD37eZ4QVzqW5AYFYN0uBSAAOMHOfTsv0PxDbR1P6L7T8XO3AslllK7jTsc6f9bqVdqCGCrfQ1nP/0SqTx40oZJarndPKeZtKep1autJxWR67lQbUmc8eBmELPkcutoYgf40vSdclf0dd+jZM5+H1vqjY1qiSo+elZPU+jmEnBfqYnhFbGv8HpxSXrZlsH+0Uxo5AjvO8GTn858xcXHPl+KKqqcMJtmM92joubF9I68d/LOGZMTg4aiXZgGvcXJzdYjT5NJ1qqE34EKunA04Ywm0y2vL5C1DL2/m3z22wWN1dRIhVV4QxmKapSNayfHTCZW2M2n+jp3r7h/3e7/4wEUEWbURLG+a+rTOI+sh4opcE4qaooU7LvKbCw1fOn4ALQUsnsD/IUkuekxZrPweGiTnUt56pMveQ1Wr+vzaPLt7JK/7nyDaXWaVL0haX8vA9kGIEbsF7Jj83phZgqsYRUUv4zY2P9DicM8hNC+a4yJAxZIXSbqIycZe/TtxANV59hY1BQ4mRSvKuWVwKoRaWOIMC5yYYyg0k8EeLhZbdLpRmUJD8EbrJ3/mK2XkEwRZLHVsOXYfFjZCnhhhcWRRUpib+VmmaoG9mocNlrY6OKynoXCinndq3Tw1mStc79jsODCzjBvfHkMNJbXk7KcD8BpBxXhAFzR7d

In [14]:
# Inspeccionar el historial completo guardado por el checkpointer
print("Mensajes en sesion-001:")
for msg in resultado_2["messages"]:
    msg.pretty_print()

Mensajes en sesion-001:
================================ Human Message =================================

Hola, me llamo Antia y soy doctora.
================================== Ai Message ==================================

[{'type': 'text', 'text': '¡Hola, Antia! Es un placer saludarte. He tomado nota de que eres doctora; es un gusto contar con esa información para asistirte mejor de ahora en adelante.\n\nComo tu asistente personal, estoy aquí para ayudarte en lo que necesites, ya sea organizando tu agenda profesional, investigando temas médicos, redactando documentos o cualquier otra tarea de tu día a día.\n\n¿En qué puedo ayudarte hoy?', 'extras': {'signature': 'EssKCsgKAb4+9vu+WtTgFrZ+VGuxUOUQo4H8+OzjxG/lDg2GlHTO6fD1AXF7kTSSwiv6hf1772gr/smO21xM3epu7+DClk2Konz/gbymnhd0XNILlsYGw0t7weHfnWgEc24gbDdMSsAT+Gj6WUVqmQups+fRzMlL5D5yKOlfHoaOF46rLtXmD3d3SXHEzoSbZ1L7SoD+/pdCfbXx49QDUTNx6Z5qhrRj9G9rjKEmGTKyKkwb8gFCpnu+INEhJudgVfIAKCQIPd+nU4hOWXewrP5RZiOGC3iZ1osehnjWGGzP7VofH0iN/ENKRESYcuSorVRVLB

Con `MemorySaver` y `thread_id`, cada conversación es **independiente y persistente** — el agente recuerda el historial completo de cada hilo sin que nosotros tengamos que gestionarlo. Distintos `thread_id` = distintas conversaciones aisladas.

## Actividad

Crea un agente con memoria persistente que actúe como un **asistente de estudio personalizado**. El agente debe:

1. Recordar el nombre del estudiante y los temas que ya estudió
2. Mantener al menos 3 turnos de conversación coherentes
3. Usar un `thread_id` propio para tu sesión

Prueba también a iniciar una segunda conversación con un `thread_id` diferente y verifica que no haya cruce de información.

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Agente con memoria persistente como asistente de estudio
mi_memoria = MemorySaver()

mi_agente = create_agent(
    ChatGoogleGenerativeAI(model="gemini-flash-latest"),
    tools=[],
    system_prompt=(
        "Eres un asistente de estudio personalizado. "
        "Recuerda el nombre del estudiante y todos los temas que ha estudiado en la sesión. "
        "Sugiere qué temas repasar o explorar a continuación según lo que haya visto. "
        "Responde siempre en español."
    ),
    checkpointer=mi_memoria,
)

mi_config = {"configurable": {"thread_id": "estudio-pablo-001"}}

# Turno 1
r1 = mi_agente.invoke(
    {"messages": [{"role": "user", "content": "Hola, me llamo Pablo. Hoy repasé álgebra lineal y empecé con cálculo diferencial."}]},
    config=mi_config,
)
print(r1["messages"][-1].content)

[{'type': 'text', 'text': '¡Hola, Pablo! Es un gusto saludarte. He anotado que hoy estuviste trabajando en **Álgebra Lineal** y que diste tus primeros pasos en **Cálculo Diferencial**. ¡Es una combinación excelente y muy potente!\n\nPara tus próximas sesiones, basándome en lo que viste hoy, te sugiero lo siguiente:\n\n1.  **Para Cálculo Diferencial (Continuación):** Como acabas de empezar, lo ideal sería asegurarte de que los conceptos de **límites y continuidad** estén bien claros, ya que son la base de la derivada. Si ya te sientes cómodo con eso, el siguiente paso lógico es la **interpretación geométrica de la derivada** (la pendiente de la recta tangente).\n2.  **Para Álgebra Lineal (Repaso):** Si hoy repasaste matrices o vectores, podrías seguir con **determinantes** o empezar a explorar los **sistemas de ecuaciones lineales**. Estos conceptos se conectarán muy pronto con lo que estás viendo en cálculo.\n3.  **Conexión entre ambos:** Más adelante, te mostraré cómo el álgebra linea

In [16]:
# Turno 2 — el agente debe recordar el nombre y los temas del turno anterior
r2 = mi_agente.invoke(
    {"messages": [{"role": "user", "content": "¿Recuerdas qué temas estudié hoy y cómo me llamo?"}]},
    config=mi_config,
)
print(r2["messages"][-1].content)

[{'type': 'text', 'text': '¡Claro que sí, no lo he olvidado!\n\nTu nombre es **Pablo** y hoy estuviste trabajando en dos áreas fundamentales de las matemáticas:\n\n1.  **Álgebra Lineal** (repaso).\n2.  **Cálculo Diferencial** (inicio de nuevos temas).\n\nComo te mencioné antes, si quieres continuar ahora o en tu próxima sesión, te sugiero que reforcemos los **límites** en cálculo (que son la base de todo) o que verifiquemos cómo vas con el cálculo de **determinantes** en álgebra lineal.\n\n¿En cuál de los dos te gustaría profundizar en este momento? ¡Estoy listo para ayudarte!', 'extras': {'signature': 'EpgLCpULAb4+9vsUy20K5hheV3ujYB4luf3OYBNdevXOg5PHOzY/x/EGRh/eoVklUpf+w2v7IbQdVY5MKM29xdCubvSVupy9Ecy3Al+aktFlKkaa3TplKAGAkUyKGNhokyAhuuNv5W5R/dgW0bUDye3Ar6m/CX11TFR+8uIvtvFiGFLIfDFxWS47Lb25u4azavwYXWECWMYqKvR/fAW9BE/3hAf8itnFXiMzZ7qVVoQNkaAWTrhVGo9WF8uuOeZF66Nn4E4WFzdarCQe5/oEXxVYH0Qnw9kOTFUCwDPJpiTjrSQVwHL8HjRFuIjMMH/Z8GIyIDSIMCHA6PvU28wEp7u8FWIqhbKmb6GY0Ppv/BXNYNrQvb3sXY8pq/7GGaogGJDyd

In [17]:
# Turno 3 — pide recomendación con base en el historial
r3 = mi_agente.invoke(
    {"messages": [{"role": "user", "content": "¿Qué tema me recomiendas estudiar mañana para complementar lo de hoy?"}]},
    config=mi_config,
)
print(r3["messages"][-1].content)

# Sesión distinta — no debe saber nada del hilo anterior
mi_config_2 = {"configurable": {"thread_id": "estudio-maria-002"}}
r_nuevo = mi_agente.invoke(
    {"messages": [{"role": "user", "content": "¿Sabes cómo me llamo o qué estudié hoy?"}]},
    config=mi_config_2,
)
print("\nNueva sesión (thread distinto):", r_nuevo["messages"][-1].content)

[{'type': 'text', 'text': 'Para complementar lo que viste hoy, **Pablo**, te sugiero dos caminos dependiendo de en qué parte específica de cada materia te hayas quedado:\n\n### 1. Para Cálculo Diferencial: **Reglas de Derivación y la Regla de la Cadena**\nSi hoy viste la definición de derivada (el límite cuando $h$ tiende a 0), mañana lo ideal es que aprendas las "atajos".\n*   **Por qué:** Te permitirá calcular derivadas de forma mucho más rápida sin usar la definición formal cada vez. Dominar la **Regla de la Cadena** es el paso más importante para no trabarte después en temas más complejos.\n\n### 2. Para Álgebra Lineal: **Sistemas de Ecuaciones Lineales (Método de Gauss-Jordan)**\nSi hoy repasaste matrices y vectores, mañana podrías enfocarte en cómo resolver sistemas de ecuaciones usando esas matrices.\n*   **Por qué:** El método de Gauss es la herramienta básica que usarás durante todo el curso para hallar bases, rangos e inversas. Además, te ayuda a desarrollar la agilidad menta